In [4]:
# =============================================================================
# EXCEL → RAW CSV (same dir) → LIGHT CLEANING → ONE EDA CSV (long-form)
# =============================================================================
# You get exactly three files next to this script:
#   1) data_raw.csv     – verbatim export (no changes)
#   2) data_clean.csv   – lightly cleaned & typed (safe for EDA)
#   3) eda_overview.csv – one long report with sections:
#        - dtype_changes           (before vs after types)
#        - yn_detection_before     (which columns looked Y/N by name)
#        - yn_inference_after      (what those columns became)
#        - columns_summary         (dtype, non-null, missing, unique, example)
#        - numeric_summary         (describe() in long format)
#        - top_values              (top N for categorical/bool cols)
#        - countries_counts        (if a 'country' column exists)
#
# Design goals:
# - Reproducible defaults that won’t mangle data (gentle typing; reversible).
# - Minimal repetition (helpers are small & focused).
# - Comments explain “why”, not only “what”.
#
# Install once (if needed):  pip install pandas numpy openpyxl
# =============================================================================

from __future__ import annotations
from pathlib import Path
from typing import Iterable
import re

import numpy as np
import pandas as pd

# ───────────────────────────── USER CONFIG ─────────────────────────────
# Path to the Excel workbook you want to process.
# • You can provide an absolute path (recommended) or path relative to the script.
# • SHEET_NAME accepts either an int (0-based; 0 = first sheet) or a string.
INPUT_XLSX   = r"C:\Users\James\OneDrive\Desktop\Files\Work\Harper\Data Work\CA23107participant list.xlsx"
SHEET_NAME   = 0          # 0 = first sheet; or use a string sheet name
TOP_N_VALUES = 25         # Number of top categories to capture per column in EDA

# Heuristic: “columns that probably mean yes/no” (by *name*, case-insensitive).
# Why names and not values? Early detection helps us log the *intent* of a field
# before we normalize values. This way, the EDA can show what changed and why.
# Patterns below catch common project-specific flags:
#   - wg1_..., wg_2_... (working groups), mc_member, core_group, is_*
YN_COLUMN_HINTS: tuple[str, ...] = (
    r"^wg\s*\d+",         # e.g., wg1, wg_2 → typical working group flags
    r"^is_",              # e.g., is_active, is_member → boolean-like fields
    r"\bmc\b|\bmc_member\b",
    r"\bcore\b|\bcore_group\b",
    r"^wg_member$",       # a single 'wg_member' flag if present
)
# Pre-compile regexes once for speed and clarity.
_YN_PATTERNS = [re.compile(p, flags=re.I) for p in YN_COLUMN_HINTS]

# ─────────────────────────── PATH RESOLUTION ──────────────────────────
# Resolve the directory to write outputs next to this script (robust in REPL too).
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:  # __file__ is undefined in notebooks/REPL → fall back to CWD
    SCRIPT_DIR = Path.cwd()

# Make the input path absolute (relative paths are resolved from the script dir).
EXCEL_PATH = Path(INPUT_XLSX)
if not EXCEL_PATH.is_absolute():
    EXCEL_PATH = (SCRIPT_DIR / EXCEL_PATH).resolve()

# Output directory = the script directory (keeps all artifacts together).
OUT_DIR = SCRIPT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ────────────────────────────── HELPERS ───────────────────────────────
def write_csv(df: pd.DataFrame, name: str) -> Path:
    """
    Save a DataFrame to UTF-8 CSV (with BOM for Excel friendliness) and return the path.

    Why UTF-8 with BOM (utf-8-sig)?
    - Opening in Excel on Windows often expects a BOM to display Unicode cleanly.
    """
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved → {path.name}")
    return path

def snake_case(name: str) -> str:
    """
    Normalize column headers into lowercase_with_underscores.

    Why do this?
    - Consistent machine-friendly names prevent subtle bugs (spaces, punctuation),
      and make downstream code more predictable (e.g., attribute-style access).
    """
    name = str(name).replace("\u00A0", " ")                  # Replace NBSP with space.
    name = re.sub(r"[^\w\s\-]+", " ", name)                  # Remove punctuation/symbols.
    name = " ".join(name.split()).strip().lower()            # Normalize whitespace & case.
    name = name.replace("-", " ")                            # Hyphens → spaces.
    return re.sub(r"\s+", "_", name)                         # Spaces → underscores.

def clean_text_series(s: pd.Series) -> pd.Series:
    """
    Gentle text clean for object columns:
      - Remove asterisks (common for footnotes)
      - Collapse multiple spaces, strip leading/trailing spaces
      - Convert empty strings and literal 'nan' → NaN

    Why gentle?
    - We avoid aggressive operations that could change meaning (e.g., case folding
      is okay here, but we keep original case because case may be meaningful).
    """
    out = (
        s.astype(str)
         .str.replace(r"\*", "", regex=True)
         .str.replace("\u00A0", " ")
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )
    return out.replace({"": np.nan, "nan": np.nan})

def looks_like_yn(names: Iterable[str]) -> list[bool]:
    """
    Heuristic classifier over *column names* only.
    Returns a parallel list of booleans for whether each name matches Y/N-ish patterns.

    Why name-based first?
    - Value-based detection can be misleading if data is messy (e.g., 'pending', 'N/A').
      We first flag candidate columns by *intent* and then report how they actually parse.
    """
    res = []
    for c in names:
        nm = str(c).lower()
        res.append(any(p.search(nm) for p in _YN_PATTERNS))
    return res

def normalize_yes_no(s: pd.Series, pending_policy: str = "false") -> pd.Series:
    """
    Map common yes/no tokens → True/False, with a clear rule for 'pending'.

    Rules:
      - Any value containing the word 'pending' ⇒ handled by pending_policy
      - 'yes' at the start of the string (e.g., 'yes', 'yes but leaving') ⇒ True
      - exact tokens {'y','yes','true','1','member'} ⇒ True
      - exact tokens {'n','no','false','0'} ⇒ False
      - everything else stays as-is (we preserve original values when uncertain)

    Why default pending→False?
    - For operational datasets, 'pending' often means "not active yet", which behaves
      like False in most analyses. You can switch to 'na' to be conservative.

    pending_policy:
      - "false" (default) : treat any 'pending' as False
      - "na"              : treat 'pending' as missing (NaN)
      - "as_is"           : leave the original text
    """
    true_tokens  = {"y", "yes", "true", "1", "member"}
    false_tokens = {"n", "no", "false", "0"}

    raw = s.astype(str).str.strip().str.lower()

    # Separate 'pending' first so it doesn't get coerced to True via 'yes' match.
    is_pending = raw.str.contains(r"\bpending\b", na=False)

    # Treat strings that *start* with 'yes' as True (captures 'yes but...' cases).
    is_yesish  = raw.str.contains(r"^\s*yes\b", na=False)

    # Compute boolean masks for true/false tokens.
    is_true  = (raw.isin(true_tokens) | is_yesish) & ~is_pending
    is_false = raw.isin(false_tokens)

    # Start with an object series so we can preserve original values for "unknown".
    out = pd.Series(index=s.index, dtype="object")
    out[is_true]  = True
    out[is_false] = False

    # Resolve 'pending' per policy.
    if pending_policy == "false":
        out[is_pending] = False
    elif pending_policy == "na":
        out[is_pending] = pd.NA
    # "as_is": leave pending values unchanged

    # Where we did not explicitly set a boolean/NA, keep the original value.
    return out.where(~out.isna(), s)

def gentle_type_infer(s: pd.Series) -> pd.Series:
    """
    Light typing in two passes (to avoid mangling free-text):
      1) numeric if ≥60% look numeric
      2) datetime if ≥50% look date-like (and ≥50% parse successfully)

    Why thresholds?
    - Mixed columns happen (e.g., "12", "13", "unknown"). We only coerce types when
      the *majority* looks consistent, ensuring analyses are stable and reversible.
    """
    # NUMERIC PASS: allow commas in thousands (we remove them before parsing).
    if s.dtype == "object":
        raw = s.astype(str).str.replace(",", "").str.strip()
        looks_num = raw.str.match(r"^-?\d+(\.\d+)?$", na=False)  # simple numeric pattern
        if looks_num.mean() >= 0.6:                              # only if ≥60% look numeric
            parsed = pd.to_numeric(raw, errors="coerce")
            return parsed

    # DATETIME PASS: safe-guard to avoid re-parsing already-datetime columns.
    if s.dtype == "object" or s.dtype.kind in "Mm":
        if s.dtype.kind in "Mm":
            return s  # already datetime-like; keep as-is
        sample = s.astype(str).str.lower()
        looks_date = sample.str.contains(
            r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
            regex=True, na=False
        )
        if looks_date.mean() >= 0.5:  # at least half look date-like
            # dayfirst=True helps with European formats like 13/10/2025
            parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.5:  # only keep if at least half parse successfully
                return parsed
    return s  # Default: preserve original dtype if criteria aren’t met.

def clean_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    If a 'country' column exists (any case), add 'country_clean' for tidy counting.

    What we do:
    - Remove parenthetical codes like "(DE)" that often trail official names
    - Apply gentle text cleaning
    - Map a few common aliases → a canonical form (e.g., 'UK' → 'United Kingdom')

    Why a new column?
    - Preserving the original raw field avoids losing fidelity while enabling clean EDA.
    """
    matches = [c for c in df.columns if c.lower() == "country"]
    if not matches:
        return df
    col = matches[0]
    df["country_clean"] = (
        df[col].astype(str)
               .str.replace(r"\(.*?\)", "", regex=True)  # drop trailing codes like “(DE)”
    )
    df["country_clean"] = clean_text_series(df["country_clean"])

    # Limited alias table: extend as needed for your domain.
    aliases = {
        "uk": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
        "czech republic": "Czechia", "turkey": "Türkiye", "turkiye": "Türkiye",
        "north macedonia": "North Macedonia", "macedonia": "North Macedonia",
        "bosnia and herzegovina": "Bosnia and Herzegovina",
        "moldova": "Moldova", "russia": "Russia",
        "ivory coast": "Côte d'Ivoire", "cote d ivoire": "Côte d'Ivoire",
        "republic of kosovo": "Kosovo", "kosovo": "Kosovo",
    }
    low = df["country_clean"].str.lower()
    df.loc[low.isin(aliases), "country_clean"] = low.map(aliases)
    return df

def value_counts_preview(s: pd.Series, n: int = 10) -> str:
    """
    Return a one-line preview of top values:
      'Yes: 20; No: 10; <NA>: 2'

    Why a string preview?
    - EDA report becomes human-scannable in a spreadsheet without pivoting.
    """
    vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
    return "; ".join(f"{k}: {int(v)}" for k, v in vc.items())

def mark_section(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Tag a DataFrame with a 'section' column so we can vertically stack many
    different EDA outputs into a single long-form CSV (eda_overview.csv).

    Why long-form?
    - Easy to filter, pivot, and share as a single artifact.
    """
    out = df.copy()
    out.insert(0, "section", name)
    return out

# ──────────────────────────────── PIPELINE ───────────────────────────────
def main() -> None:
    """
    1) Read Excel exactly as-is (df_raw)
    2) Write df_raw to data_raw.csv (verbatim reference)
    3) Detect Y/N-like columns *by name* (pre-clean snapshot)
    4) Light cleaning:
         - normalize headers
         - trim/clean text object columns
         - normalize Y/N values for Y/N-like columns
         - gentle type inference (numeric → datetime)
         - optional: create country_clean
    5) Write cleaned dataset to data_clean.csv
    6) Build a single long-form EDA CSV with multiple sections for quick review
    """
    # 1) READ EXCEL as-is (we don’t modify df_raw)
    if not EXCEL_PATH.exists():
        print(f"ERROR: Excel not found → {EXCEL_PATH}")
        return

    print(f"Reading Excel: {EXCEL_PATH}")
    try:
        df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
    except Exception as e:
        # Broad except is okay here because we want to surface any IO/engine errors clearly.
        print(f"ERROR reading Excel: {e}")
        return

    # Helpful peek for newcomers (STDOUT only; not written to files).
    print("\nRaw sample (first 5 rows):")
    print(df_raw.head())

    # 2) SAVE RAW CSV (verbatim reference copy for audits and diffing).
    write_csv(df_raw, "data_raw.csv")

    # 3) DETECT Y/N-ish COLUMNS BEFORE CLEANING (based on names only)
    #    We record this *before* renaming so stakeholders can map original headers.
    yn_flags_before = looks_like_yn(df_raw.columns)
    yn_candidates_raw = [c for c, is_yn in zip(df_raw.columns, yn_flags_before) if is_yn]
    yn_before_df = pd.DataFrame(
        {
            "original_name": yn_candidates_raw,
            "normalized_name_if_any": [snake_case(c) for c in yn_candidates_raw],
            "dtype_raw": [str(df_raw[c].dtype) for c in yn_candidates_raw],
            "top_values_preview": [value_counts_preview(df_raw[c], n=10) for c in yn_candidates_raw],
        }
    )

    # 4) LIGHT CLEANING (reproducible across datasets)
    df = df_raw.copy()

    # 4a) normalize headers to snake_case (safe for code & SQL).
    old_to_new = {c: snake_case(c) for c in df.columns}
    df.rename(columns=old_to_new, inplace=True)

    # 4b) clean all object columns (whitespace/asterisks only; gentle).
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(clean_text_series)

    # 4c) coerce likely Y/N (by cleaned names—not values—to avoid false positives).
    yn_flags_after = looks_like_yn(df.columns)
    yn_after_names = [c for c, is_yn in zip(df.columns, yn_flags_after) if is_yn]
    for c in yn_after_names:
        df[c] = normalize_yes_no(df[c], pending_policy="false")  # adjust policy if needed

    # 4d) gentle type inference (numeric then datetime, only if clearly fits).
    for c in df.columns:
        df[c] = gentle_type_infer(df[c])

    # 4e) country helper (if applicable) → adds 'country_clean' without overwriting original.
    df = clean_country_column(df)

    print("\nCleaned sample (first 5 rows):")
    print(df.head())

    # 5) SAVE CLEANED CSV (ready for EDA, BI tools, or modeling experiments).
    write_csv(df, "data_clean.csv")

    # 6) BUILD ONE LONG-FORM EDA CSV
    eda_parts: list[pd.DataFrame] = []

    # 6a) dtype changes (before vs after) → quick sanity check on what changed.
    dtype_changes = pd.DataFrame({
        "original_name": list(df_raw.columns),
        "cleaned_name": [old_to_new.get(c, c) for c in df_raw.columns],
        "dtype_raw": [str(df_raw[c].dtype) for c in df_raw.columns],
        "dtype_final": [str(df[old_to_new.get(c, c)].dtype) if old_to_new.get(c, c) in df.columns else "<missing>" for c in df_raw.columns],
    })
    dtype_changes["changed"] = dtype_changes["dtype_raw"] != dtype_changes["dtype_final"]
    eda_parts.append(mark_section(dtype_changes, "dtype_changes"))

    # 6b) Y/N detection before cleaning (captures *intent* by original names).
    if not yn_before_df.empty:
        eda_parts.append(mark_section(yn_before_df, "yn_detection_before"))

    # 6c) Y/N inference after cleaning (what they became and counts).
    if yn_candidates_raw:
        raw_to_clean = {raw: old_to_new.get(raw, raw) for raw in yn_candidates_raw}
        rows = []
        for raw, cleaned in raw_to_clean.items():
            if cleaned in df.columns:
                s = df[cleaned]
                looks_bool = (s.dtype == bool) or s.dropna().isin([True, False]).all()
                rows.append({
                    "original_name": raw,
                    "cleaned_name": cleaned,
                    "final_dtype": str(s.dtype),
                    "true_count": int((s == True).sum()) if looks_bool else None,    # noqa: E712
                    "false_count": int((s == False).sum()) if looks_bool else None,  # noqa: E712
                    "na_count": int(s.isna().sum()),
                    "top_values_after": value_counts_preview(s, n=10),
                })
            else:
                # In case a column got dropped/renamed unexpectedly.
                rows.append({
                    "original_name": raw,
                    "cleaned_name": cleaned,
                    "final_dtype": "<missing>",
                    "true_count": None, "false_count": None, "na_count": None,
                    "top_values_after": "",
                })
        eda_parts.append(mark_section(pd.DataFrame(rows), "yn_inference_after"))

    # 6d) columns summary: per-column shape & flavor for quick profiling.
    cols_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        # Example value helps human QA to understand content at a glance.
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    }).sort_values("column")
    eda_parts.append(mark_section(cols_summary, "columns_summary"))

    # 6e) numeric summary (describe → long format) for easy pivoting/filtering in Excel.
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        desc_long = (
            df[num_cols].describe(include="all")
                        .T.reset_index()
                        .rename(columns={"index": "column"})
                        .melt(id_vars="column", var_name="metric", value_name="value")
        )
        eda_parts.append(mark_section(desc_long, "numeric_summary"))

    # 6f) top values for categorical/bool/category columns (up to TOP_N_VALUES).
    cat_cols = df.select_dtypes(include=["object", "bool", "category"]).columns
    if len(cat_cols):
        rows = []
        for c in cat_cols:
            vc = df[c].astype("object").fillna("<NA>").value_counts(dropna=False).head(TOP_N_VALUES)
            rows.extend({"column": c, "value": k, "count": int(v)} for k, v in vc.items())
        eda_parts.append(mark_section(pd.DataFrame(rows), "top_values"))

    # 6g) country counts (if present), prefer 'country_clean' for tidy aggregation.
    country_col = [c for c in df.columns if c == "country_clean"] or [c for c in df.columns if c == "country"]
    if country_col:
        country_counts = (
            df[country_col[0]].astype("object").fillna("<NA>")
              .value_counts(dropna=False)
              .rename_axis("country")
              .reset_index(name="count")
        )
        eda_parts.append(mark_section(country_counts, "countries_counts"))

    # 6h) write the single EDA CSV (long-form with 'section' label).
    eda_overview = pd.concat(eda_parts, ignore_index=True, sort=False) if eda_parts else pd.DataFrame({"section":[]})
    write_csv(eda_overview, "eda_overview.csv")

    print("\nAll files written to:", OUT_DIR.resolve())

# ───────────────────────────────── RUN ────────────────────────────────
if __name__ == "__main__":
    main()


Reading Excel: C:\Users\James\OneDrive\Desktop\Files\Work\Harper\Data Work\CA23107participant list.xlsx

Raw sample (first 5 rows):
          Country Core Group Core Group Member Title MC Member WG Member  \
0    Türkiye (TR)        No                      NaN    Member       Yes   
1    Romania (RO)        No                      NaN        No       Yes   
2  Lithuania (LT)        No                      NaN    Member       Yes   
3  Lithuania (LT)        No                      NaN        No       Yes   
4    Germany (DE)        Yes              WG4 Leader    Member       Yes   

  WG1. Setting the scene: Assessing the current state of evidence synthesis in agri-food.  \
0                                                  y                                        
1                                                  n                                        
2                                                  y                                        
3                                      

C:\Users\James\AppData\Local\Temp\ipykernel_44852\1097465598.py:261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
C:\Users\James\AppData\Local\Temp\ipykernel_44852\1097465598.py:261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
C:\Users\James\AppData\Local\Temp\ipykernel_44852\1097465598.py:261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a futu